In [ ]:
#part of the code please refer to https://github.com/AI4Finance-Foundation/FinRL 
!pip install git+https://github.com/AI4Finance-Foundation/FinRL.git

  Cloning https://github.com/AI4Finance-Foundation/FinRL.git to /tmp/pip-req-build-tx05qqcl
  Running command git clone --filter=blob:none --quiet https://github.com/AI4Finance-Foundation/FinRL.git /tmp/pip-req-build-tx05qqcl
  Resolved https://github.com/AI4Finance-Foundation/FinRL.git to commit d25d902a6de54931a329adc38a2663e8f576adc4
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
  Cloning https://github.com/AI4Finance-Foundation/ElegantRL.git to /tmp/pip-install-20v3q8uv/elegantrl_047e595d777b45e3aef8aa3ecf984158
  Running command git clone --filter=blob:none --quiet https://github.com/AI4Finance-Foundation/ElegantRL.git /tmp/pip-install-20v3q8uv/elegantrl_047e595d777b45e3aef8aa3ecf984158
  Resolved https://github.com/AI4Finance-Foundation/ElegantRL.git to commit 5e828af1503098f4da046c0f12432dbd4ef8bd97
  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 108.7/1

data prepare and index calculation

In [36]:
import pandas as pd
import yfinance as yf

from finrl.meta.preprocessor.yahoodownloader import YahooDownloader
from finrl.meta.preprocessor.preprocessors import FeatureEngineer, data_split
from finrl import config_tickers
from finrl.config import INDICATORS
from finrl.config import *
import itertools

In [ ]:
TRAIN_START_DATE = '2009-01-01'
TRADE_END_DATE = '2025-03-31'
aapl_df_yf = yf.download(tickers = "aapl", start=TRAIN_START_DATE, end=TRADE_END_DATE)

[*********************100%***********************]  1 of 1 completed


In [82]:
aapl_df_finrl = YahooDownloader(start_date = TRAIN_START_DATE,
                                end_date = TRAIN_END_DATE,
                                ticker_list = ['aapl']).fetch_data()

[*********************100%***********************]  1 of 1 completed

Shape of DataFrame:  (2893, 8)


In [87]:
stock_tickers = [
    'AAPL', 'MSFT', 'GOOGL', 'AMZN', 'META', 'TSLA', 'NVDA', 'NFLX', 'ADBE', 'INTC',
    'ORCL', 'IBM', 'CRM', 'PYPL', 'QCOM', 'TXN', 'AVGO', 'AMD', 'SBUX', 'UBER'
]


In [88]:
TRAIN_START_DATE = '2009-01-01'
TRAIN_END_DATE = '2020-07-01'
TRADE_START_DATE = '2020-07-01'
TRADE_END_DATE = '2025-03-31'

In [89]:
df_raw = YahooDownloader(start_date = TRAIN_START_DATE,
                     end_date = TRADE_END_DATE,
                     ticker_list = stock_tickers).fetch_data()

[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed
[*********************100%********

Shape of DataFrame:  (76085, 8)


In [91]:
fe = FeatureEngineer(use_technical_indicator=True,
                     tech_indicator_list = INDICATORS,
                     use_vix=True,
                     use_turbulence=True,
                     user_defined_feature = False)

processed = fe.preprocess_data(df_raw)

[*********************100%***********************]  1 of 1 completed

Successfully added technical indicators
Shape of DataFrame:  (4084, 8)
Successfully added vix


Successfully added turbulence index


In [92]:
list_ticker = processed["tic"].unique().tolist()
list_date = list(pd.date_range(processed['date'].min(),processed['date'].max()).astype(str))
combination = list(itertools.product(list_date,list_ticker))

processed_full = pd.DataFrame(combination,columns=["date","tic"]).merge(processed,on=["date","tic"],how="left")
processed_full = processed_full[processed_full['date'].isin(processed['date'])]
processed_full = processed_full.sort_values(['date','tic'])

processed_full = processed_full.fillna(0)

In [93]:
processed_full.head()

,date,tic,close,high,low,open,volume,day,macd,boll_ub,boll_lb,rsi_30,cci_30,dx_30,close_30_sma,close_60_sma,vix,turbulence
0,2009-01-02,AAPL,2.730994,3.251429,3.041429,3.067143,746015200.0,4.0,0.0,2.951624,2.625623,100.0,66.666667,100.0,2.730994,2.730994,39.189999,0.0
1,2009-01-02,ADBE,23.020000,23.100000,21.070000,21.110001,6670700.0,4.0,0.0,2.951624,2.625623,100.0,66.666667,100.0,23.020000,23.020000,39.189999,0.0
2,2009-01-02,AMD,2.380000,2.430000,2.170000,2.190000,13832100.0,4.0,0.0,2.951624,2.625623,100.0,66.666667,100.0,2.380000,2.380000,39.189999,0.0
3,2009-01-02,AMZN,2.718000,2.726500,2.553500,2.567500,145928000.0,4.0,0.0,2.951624,2.625623,100.0,66.666667,100.0,2.718000,2.718000,39.189999,0.0
4,2009-01-02,CRM,8.444491,8.550000,7.912500,8.025000,4069200.0,4.0,0.0,2.951624,2.625623,100.0,66.666667,100.0,8.444491,8.444491,39.189999,0.0


In [94]:
train = data_split(processed_full, TRAIN_START_DATE,TRAIN_END_DATE)
trade = data_split(processed_full, TRADE_START_DATE,TRADE_END_DATE)
print(len(train))
print(len(trade))

43395
17865


In [95]:
train.to_csv('train_data.csv')
trade.to_csv('trade_data.csv')